# Fund time series data enrichment & quality check

**Approach :**

- Load the **Timeseries** sheet of `CLEAN_RECOLTE DONNEES-EXCEL.xlsx`
- Build the fund list (ISIN, name, wrapper, type) and map each ISIN to a market identifier from `cache-isin-openfigi.json`
- Set `SELECTED_FUND` to the ISIN to check
- Load one frame on the Lipper observation dates: the fund’s monthly total return from the sheet (**Lipper**), the same fund from Yahoo and from FinanceDatabase, plus **S&P 500 TR** from `cache-sp500-tr-indices.json` and **US T-Bill 3M** from the sheet
- Compare monthly returns of the master series (default Lipper) with the comparison series (default Yahoo)
- Check each series for gaps, abnormal spikes, scale changes, and months far from the S&P 500 TR
- Regress the master return on S&P 500 TR, then compare alpha, beta, R², and tracking error on 3-, 5-, and 10-year windows with the workbook **Funds** sheet, and compare beta, alpha, tracking error, and related stats with the **Consolidation** sheet (fixed AE6–AE5 window)
- For the largest master–comparison gap, test nearby Yahoo sessions to see whether a shifted observation date would reconcile the return
- Ask OpenAI to score the check and write a short interpretation of the timeseries and the regression
- Save the run under `output-data/timeseries-checks/<ISIN>`


### How quality of a return timeseries is verified

The Timeseries sheet stores monthly total returns but **no per-fund NAV date**, so observation dates are taken from the S&P 500 TR calendar in `cache-sp500-tr-indices.json` (resolved in `check-s&p500-timeseries.ipynb` by matching Lipper index levels to Yahoo). That calendar is the default **`Date`** for every fund. In some months Lipper dated a fund's NAV on a nearby session instead (for example **Dec 2021**: index `2021-12-01` while many funds were dated `2021-11-30`). Before any comparison, each fund-month is checked against Yahoo total-return levels: if the index date already reproduces Lipper within tolerance, it is kept; otherwise a shift of at most six sessions is accepted only when Yahoo **exactly** reproduces Lipper on that window. Genuine disagreements (distributions, bad values) keep the index date so they stay visible. Fund, Yahoo, FinanceDatabase, and **S&P 500 TR** on a row then share the same resolved window, which removes artificial regression residuals from mismatched month boundaries.

**Cross-source agreement** — Lipper vs Yahoo on those resolved dates. Each month is classified as a perfect match (identical or both missing), an acceptable match (absolute gap ≤ `MATCH_TOLERANCE`, default 1 bp), or a mismatch.

**Internal consistency** — Lipper and Yahoo are checked **separately** against **S&P 500 TR** on the same frame for gaps, abnormal spikes, scale shifts, index divergence, and extremes.

**Versus the benchmark** — OLS of the master fund return on **S&P 500 TR** (alpha, beta, R², correlation, information ratio) and charts of returns and the fitted line.

**Regression vs Lipper** (`compare_regression_to_lipper_funds`) — the same OLS on 3-, 5-, and 10-year windows compared to the workbook **Funds** sheet.

**Regression vs Consolidation** (`compare_regression_to_consolidation`) — OLS on the fixed Consolidation window (**AE6–AE5**) compared to the **Consolidation** sheet row (sheet replication, not external validation).

**Worst mismatch drill-down** — runs **after** date resolution on the largest remaining gap; nearby Yahoo sessions (±6) test whether a further shift would reconcile Lipper (usually indicating a genuine disagreement rather than calendar dating).


In [2]:
import sys
from pathlib import Path
import pandas as pd


def _ensure_helpers_importable() -> None:
    """Put code/ on sys.path whether the kernel cwd is code/ or notebooks/."""
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "helpers").is_dir():
            root = str(directory)
            if root not in sys.path:
                sys.path.insert(0, root)
            return
    raise FileNotFoundError(f"Could not find helpers/, starting from {start}")

_ensure_helpers_importable()

from helpers.timeseries_functions.load_workbook import load_workbook


FILENAME = "CLEAN_RECOLTE DONNEES-EXCEL.xlsx"
RELATIVE = Path("input-data-alix") / "clean"

timeseries_df = load_workbook(RELATIVE, FILENAME)

print("Read the worksheet")
print(f"Number of timeseries found: {len(timeseries_df)}")

Read the worksheet
Number of timeseries found: 207


### Match with OPENFIGI and retrieve market identifier

In [3]:
from helpers.timeseries_functions.load_market_identifiers import load_market_identifiers

CACHE_RELATIVE = Path("cache") / "cache-isin-openfigi.json"

funds = load_market_identifiers(timeseries_df, CACHE_RELATIVE)


Read the list of funds - found 550 funds
Mapped to market identifiers
- Funds with market identifier: 549
- Funds without market identifier: 1


### Select fund to analyze


In [4]:
SELECTED_FUND = "US4613088016"

### Load an normalize the timeseries
 
We load the Lipper timeries, normalize the periods, compute returns using prices and load equivalent timeseries from yahoo finance and financedatabase. We add the S&P 500 and 3M T-Bill timeries.
 
**How a return is read**
 
**Note:** Yahoo's adjusted close reinvests a distribution at the previous NAV. The Yahoo series in this notebook is rebuilt so each distribution is reinvested at the ex-date close, the market convention used by Lipper and by the S&P 500 total return. 
 
Conventions:
- Each row is one month.
- Every return column is the simple total return from the **previous row's `Date`** to **this row's `Date`**: `value(this Date) / value(previous Date) − 1`
- The number is a decimal: `0.0276` means +2.76%.
- `Jan 2010` is empty because there is no earlier observation.
 
The index columns are:
- **`Period`** — month label (`Jan 2010`, `Feb 2010`, …)
- **`Date`** — **resolved observation date** for this fund-month (default = index calendar in **`Index Date`**). Shifts of at most six sessions apply only when Yahoo reproduces Lipper's return on that window.
- **`Index Date`** — index-derived observation date from `cache-sp500-tr-indices.json` before per-fund resolution.
- **`Date shift (sessions)`** — signed session count from **`Index Date`** to **`Date`** (0 when unchanged).
- **`Index source`** — `Lipper cache` when **S&P 500 TR** uses Lipper levels on the index calendar; `Yahoo ^SP500TR (shifted window)` when the index return is recomputed from Yahoo closes over the resolved window.
- **`Time`** — always **`Close`**
 
The return columns are:
- **`S&P 500 TR`** — on unshifted windows, the Lipper cache level between consecutive **index** dates; when either boundary was shifted for this fund, the return from Yahoo **`^SP500TR`** closes between consecutive **resolved** `Date` values (`Index source` marks which rule applied).
- **`US T-Bill 3M`** — the risk-free rate proxy: monthly return from the 3-month US Treasury bill index, aligned to the fund's observation dates. Sourced from the relevant database or data provider and matched by `Date` to each row.
- **`ISIN (Lipper)`** — the fund's monthly total return already stored in the Timeseries sheet ("Monthly TR return").
- **`ISIN (Yahoo)`** — the fund's market identifier on Yahoo. Each distribution is reinvested at that day's unadjusted close, `(P_ex + D) / P_prev`, the same convention as the S&P 500 total return. The column is the change in that level between observation dates.
- **`ISIN (FinanceDatabase)`** — the same fund in the FinanceDatabase catalogue. The price is the dividend-adjusted close from FinanceToolkit on that same `Date` (Financial Modeling Prep, then Yahoo Finance when FMP has no history).


In [ ]:
from helpers.timeseries_functions.load_fund_all_timeseries import (
    INDEX_COLUMN,
    load_fund_all_timeseries,
)

SP_CACHE_RELATIVE = Path("cache") / "cache-sp500-tr-indices.json"
DATE_RESOLUTION_SESSIONS = 6

fund_timeseries, selected, load_ctx = load_fund_all_timeseries(
    timeseries_df,
    funds,
    SELECTED_FUND,
    SP_CACHE_RELATIVE,
    match_tolerance=1e-4,
    max_shift_sessions=DATE_RESOLUTION_SESSIONS,
)
index_column = INDEX_COLUMN


In [ ]:
fund_timeseries.loc[
    fund_timeseries["Date shift (sessions)"] != 0,
    ["Period", "Index Date", "Date", "Date shift (sessions)", "Index source"],
]


### Compare the fund timeseries with yahoo timeseries

Lipper is compared to Yahoo monthly total returns on **resolved** observation dates (after per-fund date alignment). Remaining mismatches are, by construction, **not** explainable by a ≤6-session date shift. Typical causes: a distribution not reinvested or misdated in the Lipper export, a stale Yahoo NAV, or an isolated bad value on either side.

Each month is classified as perfect, acceptable (≤ `MATCH_TOLERANCE`), or mismatch. Stale-NAV notes in the printed report flag unchanged Yahoo closes with no distribution while the S&P 500 moved materially on that session.


In [ ]:
from helpers.timeseries_functions.compare_timeseries_sources import (
    compare_timeseries_sources,
)

# Master is the series the other one is judged against.
# Comparison is the series placed next to it.
MASTER_TIMESERIES = "Lipper"
COMPARISON_TIMESERIES = "Yahoo"

# One basis point. A gap of zero is a perfect match. A wider gap up to
# this tolerance is an acceptable match. Anything beyond it is a mismatch.
MATCH_TOLERANCE = 1e-4

compared, master_column, comparison_column, gap, mismatch = compare_timeseries_sources(
    fund_timeseries,
    SELECTED_FUND,
    selected,
    master_timeseries=MASTER_TIMESERIES,
    comparison_timeseries=COMPARISON_TIMESERIES,
    match_tolerance=MATCH_TOLERANCE,
    yahoo_level=load_ctx.yahoo_level,
    close=load_ctx.close,
    distribution=load_ctx.distribution,
)


### Check for gaps, abnormal spikes, scale changes, and months far from the S&P 500 TR

The quality check is made on both the Lipper (master) and Yahoo (comparison) timeseries. 
It inspects for data issues that could signal inaccurate or unreliable timeseries values. 
Specifically, it checks for:
- Large or abnormal monthly return spikes that may indicate errors or outliers.
- Abrupt reversals in returns, which can point to inconsistencies or reporting issues.
- Sudden, significant changes in the scale (level) of the timeseries across rolling windows, which may reflect splits, currency changes, or missing adjustments.
- Large one-time jumps in the price level.
- Months where the fund return is abnormally distant from the S&P 500 total return, which can reveal benchmark or classification problems.
- Long gaps with no data, which break continuity.
These checks help flag data problems that should be reviewed and addressed before using the timeseries for further analysis.

In [10]:
from helpers.timeseries_functions.check_timeseries_consistency import (
    check_timeseries_consistency,
)

# Spikes, gaps, scale changes, and months far from the S&P 500 TR.
# The master and the comparison are judged separately.
SPIKE_RETURN = 0.10
REVERSAL_BAND = 0.15
SCALE_FACTOR = 5
SCALE_WINDOW = 12
LEVEL_JUMP = 0.50
INDEX_GAP = 0.05
DATE_GAP_DAYS = 45

check_timeseries_consistency(
    fund_timeseries,
    index_column,
    master_column,
    comparison_column,
    master_timeseries=MASTER_TIMESERIES,
    comparison_timeseries=COMPARISON_TIMESERIES,
    spike_return=SPIKE_RETURN,
    reversal_band=REVERSAL_BAND,
    scale_factor=SCALE_FACTOR,
    scale_window=SCALE_WINDOW,
    level_jump=LEVEL_JUMP,
    index_gap=INDEX_GAP,
    date_gap_days=DATE_GAP_DAYS,
)


Lipper
    Gaps: none
    Abnormal spikes: none
    Scale shifts: none
    Differences with S&P 500 TR above 5%: none
    Largest move: 2020-04-01  Apr 2020  -17.53%  S&P 500 TR -19.91%
    Largest gap versus S&P 500 TR: 2022-08-01  Aug 2022  -269.2 bp  fund +5.08%  index +7.77%

Yahoo
    Gaps: none
    Abnormal spikes: none
    Scale shifts: none
    Differences with S&P 500 TR above 5%: none
    Largest move: 2020-04-01  Apr 2020  -17.53%  S&P 500 TR -19.91%
    Largest gap versus S&P 500 TR: 2022-08-01  Aug 2022  -269.2 bp  fund +5.08%  index +7.77%



In [12]:
import importlib

import helpers.timeseries_functions.check_timeseries_consistency as consistency

# The check cell above already imported this module. Reload it so this
# session sees locate_timeseries_issues, which was added after that import.
importlib.reload(consistency)

from helpers.timeseries_functions.check_timeseries_consistency import (
    locate_timeseries_issues,
)
from helpers.timeseries_functions.plot_timeseries_issues import plot_timeseries_issues

# Same rules and thresholds as the consistency check above.
# Each panel is one source against the S&P 500 TR.
# Gaps and scale shifts are vertical rules. Spikes, level jumps, and months
# far from the index are points on that source's return.
consistency = locate_timeseries_issues(
    fund_timeseries,
    index_column,
    master_column,
    comparison_column,
    master_timeseries=MASTER_TIMESERIES,
    comparison_timeseries=COMPARISON_TIMESERIES,
    spike_return=SPIKE_RETURN,
    reversal_band=REVERSAL_BAND,
    scale_factor=SCALE_FACTOR,
    scale_window=SCALE_WINDOW,
    level_jump=LEVEL_JUMP,
    index_gap=INDEX_GAP,
    date_gap_days=DATE_GAP_DAYS,
)

plot_timeseries_issues(consistency, index_column)


alt.VConcatChart(...)

### Compute the regression between the fund and the index

Compute a regression analysis between the fund's timeseries and the benchmark index.
It fits an OLS model to estimate Alpha, Beta, and other statistics, then displays a summary of the regression fit and associated charts.

In [13]:
from helpers.timeseries_functions.compute_fund_to_index_regression import (
    compute_fund_to_index_regression,
)

regression_result = compute_fund_to_index_regression(
    fund_timeseries,
    index_column,
    master_column,
    fund_name=selected["Name"],
    master_timeseries=MASTER_TIMESERIES,
)

regression_sample = regression_result.regression_sample
regression = regression_result.regression
coefficient_table = regression_result.coefficient_table
fit_table = regression_result.fit_table
alpha = regression_result.alpha
beta = regression_result.beta
r_squared = regression_result.r_squared
n_months = regression_result.n_months

display(coefficient_table)
display(fit_table)

regression_result.returns_chart & regression_result.regression_chart


American Funds Investment Co of America;529-E
Lipper = -0.06% + 0.927 × S&P 500 TR
201 months, 2010-02-01 to 2026-10-01. Alpha is -5.7 bp per month (-0.69% annualized, × 12).


,Estimate,Std. error,t,p-value,95% low,95% high
Parameter,,,,,,
Alpha,-0.0005731900080072197,0.0005625101372813789,-1.0189860946817009,0.3094,-0.0016824355544271868,0.0005360555384127476
Beta,0.927453487108316,0.01227110998239757,75.58024403975777,< 0.0001,0.903255391473294,0.9516515827433379


R²                                   0.966
Adjusted R²                          0.966
Correlation                          0.983
Number of months                       201
Residual standard error              0.77%
Alpha, annualized (× 12)            -0.69%
Annualized excess return            -1.75%
Annualized residual volatility       2.88%
Information ratio                    -0.61
F statistic                        5,712.4
Prob (F)                          < 0.0001
Name: Value, dtype: str

alt.VConcatChart(...)

### Compare with Lipper **Funds** sheet

The **Funds** sheet in the same workbook stores Lipper’s precomputed **Alpha**, **Beta**, **R-Squared**, and **annualised tracking error** for 3, 5, and 10 years to last month end (vs **Benchmark Name**, usually S&P 500 TR).

The table below reruns the same OLS on each horizon using our Timeseries returns, then compares to those Lipper columns. **All available** is the full regression from the cell above (no Lipper row for that window). Lipper **Alpha** is monthly percent (÷100 → decimal return); tracking error is annualized percent.


In [8]:
from helpers.timeseries_functions.compare_regression_to_lipper_funds import (
    compare_regression_to_lipper_funds,
)

lipper_regression_comparison = compare_regression_to_lipper_funds(
    regression_result,
    fund_timeseries,
    index_column,
    master_column,
    SELECTED_FUND,
    RELATIVE,
    FILENAME,
    fund_name=selected["Name"],
    master_timeseries=MASTER_TIMESERIES,
)

display(lipper_regression_comparison)


Lipper Funds benchmark: S&P 500 TR
Compare fixed horizons (3y / 5y / 10y) to Lipper columns on the Funds sheet. Lipper Alpha is stored as monthly percent (÷100 → decimal); tracking error is annualized percent. Same OLS as above, no risk-free rate.


,Horizon,Months,Metric,Notebook,Lipper Funds,Delta (notebook − Lipper)
0,All available,201,Alpha (monthly),-0.0573%,,
1,All available,201,"Alpha (annualized, ×12)",-0.69%,,
2,All available,201,Beta,0.927,,
3,All available,201,R²,0.966,,
4,All available,201,Correlation,0.983,,
5,All available,201,Annualized excess return,-1.75%,,
6,All available,201,Annualized tracking error,2.88%,,
7,All available,201,Information ratio,-0.61,,
8,3y to last month end,37,Alpha (monthly),0.0621%,0.0800%,-0.0179%
9,3y to last month end,37,Beta,0.975,0.970,0.005


### Compare with **Consolidation** sheet

The **Consolidation** sheet stores one row per fund (**Fund** = ISIN): **Beta**, **Net alpha (%/yr)**, **Gross alpha (%/yr)**, **Tracking error (%)**, **Excess return (%/yr)**, and **Information ratio**, computed from the **Timeseries** sheet over the date window **AE6** (start) through **AE5** (end).

The table below reruns the same OLS on that window using our loaded Lipper and S&P 500 TR returns. The full-sample regression in the cell above may differ if your history extends before the Consolidation start date. In the batch Excel export, the **Check: vs Consolidation** result (**Recreates sheet** / **Drifts from sheet**) measures whether this notebook reproduces the Consolidation tab—not whether Lipper matches an external data source.


In [16]:
from helpers.timeseries_functions.compare_regression_to_consolidation import (
    compare_regression_to_consolidation,
)

consolidation_regression_comparison = compare_regression_to_consolidation(
    regression_result,
    fund_timeseries,
    index_column,
    master_column,
    SELECTED_FUND,
    RELATIVE,
    FILENAME,
    fund_name=selected["Name"],
    master_timeseries=MASTER_TIMESERIES,
)

display(consolidation_regression_comparison)


Consolidation window: 2016-10-01 to 2026-10-01 (121 months).
Consolidation uses Lipper fund returns from Timeseries vs the index column (Timeseries column E in Excel; here the notebook index column). Notebook OLS is rerun on the same date window. The regression cell above uses all available months unless the window matches.


/Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,Window,Months,Metric,Notebook,Consolidation,Delta (notebook − Consolidation)
0,2016-10-01 to 2026-10-01,121,Months,121,121,0
1,2016-10-01 to 2026-10-01,121,Beta,0.909573,0.909573,0.000000
2,2016-10-01 to 2026-10-01,121,Tracking error (annualized),3.1460%,3.1460%,0.0000%
3,2016-10-01 to 2026-10-01,121,Excess return (annualized),-1.6760%,-1.6760%,0.0000%
4,2016-10-01 to 2026-10-01,121,Net alpha (annualized),-0.2702%,-0.2702%,-0.0000%
5,2016-10-01 to 2026-10-01,121,Gross alpha (annualized),0.5698%,0.5698%,-0.0000%
6,2016-10-01 to 2026-10-01,121,Information ratio,-0.532734,-0.532734,0.000000


### Diagnose worst Lipper–Yahoo mismatch

When the cross-source comparison flags mismatches, this step focuses on the **largest gap** between the master series (Lipper) and Yahoo. Using Yahoo daily levels, it tries nearby sessions (±6 days) on the **start** and **end** dates of that month’s return to see whether shifting the observation date would bring Yahoo within `MATCH_TOLERANCE` of Lipper—a **date mismatch**—or whether the sources genuinely disagree.


### Worst mismatch drill-down

Runs **after** per-fund observation-date resolution. When mismatches remain, nearby Yahoo sessions usually indicate a **genuine** Lipper–Yahoo disagreement (distribution, stale NAV, or bad value), not a calendar boundary effect.


In [ ]:
from helpers.timeseries_functions.diagnose_worst_mismatch import (
    diagnose_worst_mismatch,
)

date_check = diagnose_worst_mismatch(
    fund_timeseries,
    compared,
    gap,
    mismatch,
    master_column,
    comparison_column,
    selected,
    SELECTED_FUND,
    master_timeseries=MASTER_TIMESERIES,
    comparison_timeseries=COMPARISON_TIMESERIES,
    match_tolerance=MATCH_TOLERANCE,
    yahoo_level=load_ctx.yahoo_level,
)

date_check


### OpenAI evaluation

Before saving the check folder, ask OpenAI to score the results and interpret the quality checks and regression. The JSON response is written to `outputs/llm-evaluation-openai.json` when the save cell runs.

In [10]:
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

import importlib

import helpers.timeseries_functions.fund_llm_dossier as _fund_llm_dossier
import helpers.timeseries_functions.fund_llm_providers as _fund_llm_providers

importlib.reload(_fund_llm_dossier)
importlib.reload(_fund_llm_providers)

build_dossier_from_run = _fund_llm_dossier.build_dossier_from_run
DEFAULT_GENERATIVE_MODELS = _fund_llm_providers.DEFAULT_GENERATIVE_MODELS
call_openai = _fund_llm_providers.call_openai
provider_result_as_json = _fund_llm_providers.provider_result_as_json


OPENAI_LLM_MODEL = DEFAULT_GENERATIVE_MODELS["openai"]

llm_dossier = build_dossier_from_run(
    selected_fund=SELECTED_FUND,
    selected=selected,
    fund_timeseries=fund_timeseries,
    index_column=index_column,
    master_column=master_column,
    comparison_column=comparison_column,
    master_timeseries=MASTER_TIMESERIES,
    comparison_timeseries=COMPARISON_TIMESERIES,
    match_tolerance=MATCH_TOLERANCE,
    compared=compared,
    gap=gap,
    mismatch=mismatch,
    regression_result=regression_result,
    lipper_regression_comparison=lipper_regression_comparison,
    consolidation_regression_comparison=consolidation_regression_comparison,
    date_check=date_check,
    spike_return=SPIKE_RETURN,
    reversal_band=REVERSAL_BAND,
    scale_factor=SCALE_FACTOR,
    scale_window=SCALE_WINDOW,
    level_jump=LEVEL_JUMP,
    index_gap=INDEX_GAP,
    date_gap_days=DATE_GAP_DAYS,
    workbook_relative=RELATIVE,
    workbook_filename=FILENAME,
    shifted_months=load_ctx.shifted_months,
)

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set in the environment or .env")

openai_llm_result = call_openai(
    llm_dossier,
    api_key=api_key,
    model=OPENAI_LLM_MODEL,
)
openai_llm_evaluation = provider_result_as_json(openai_llm_result)

if openai_llm_result.ok:
    print(f"OpenAI ({openai_llm_result.model})")
    for key, value in openai_llm_result.scores.items():
        print(f"  {key}: {value:.2f}")
    print("\nTimeseries interpretation:")
    print(openai_llm_result.timeseries_interpretation)
    print("\nRegression interpretation:")
    print(openai_llm_result.regression_interpretation)
else:
    raise RuntimeError(openai_llm_result.error)


OpenAI (gpt-5.4)
  price_timeseries_reliable: 0.95
  tracks_sp500_tr: 0.22
  active_management: 0.84
  regression_quality: 0.72

Timeseries interpretation:
The monthly return series looks clean and usable. Both Lipper and Yahoo show no gaps, spikes, or scale problems, and almost all cross-source observations match within tolerance; the small exceptions are limited and one is directly attributable to a dating issue rather than a return-calculation defect.

Regression interpretation:
The fund moves closely with the S&P 500 TR, but not like a tight index tracker. High R² and correlation indicate strong market exposure, while beta below 1, annualized tracking error of 2.88%, and repeated 200+ bp monthly deviations from the index point to materially different return behaviour from passive replication.


In [11]:
import importlib

import helpers.timeseries_functions.timeseries_check_overview as _check_overview
import helpers.timeseries_functions.timeseries_check_bundle as _check_bundle
import helpers.timeseries_functions.save_timeseries_check as _save_timeseries_check

# Reload dependencies first so manifest FILE_NOTES matches the on-disk helpers.
importlib.reload(_check_overview)
importlib.reload(_check_bundle)
importlib.reload(_save_timeseries_check)
save_timeseries_check = _save_timeseries_check.save_timeseries_check

# One folder per ISIN under output-data/timeseries-checks, replaced on each run.
# overview.md is the reading order for an agent that reviews the check or drafts a report.
saved_dir = save_timeseries_check(
    selected_fund=SELECTED_FUND,
    selected=selected,
    fund_timeseries=fund_timeseries,
    index_column=index_column,
    master_timeseries=MASTER_TIMESERIES,
    comparison_timeseries=COMPARISON_TIMESERIES,
    match_tolerance=MATCH_TOLERANCE,
    compared=compared,
    master_column=master_column,
    comparison_column=comparison_column,
    gap=gap,
    mismatch=mismatch,
    regression_result=regression_result,
    lipper_regression_comparison=lipper_regression_comparison,
    consolidation_regression_comparison=consolidation_regression_comparison,
    date_check=date_check,
    spike_return=SPIKE_RETURN,
    reversal_band=REVERSAL_BAND,
    scale_factor=SCALE_FACTOR,
    scale_window=SCALE_WINDOW,
    level_jump=LEVEL_JUMP,
    index_gap=INDEX_GAP,
    date_gap_days=DATE_GAP_DAYS,
    date_resolution={
        "match_tolerance": MATCH_TOLERANCE,
        "max_shift_sessions": DATE_RESOLUTION_SESSIONS,
    },
    shifted_months=load_ctx.shifted_months,
    workbook_relative=RELATIVE,
    workbook_filename=FILENAME,
    openai_llm_evaluation=openai_llm_evaluation,
)

saved_dir

Saved timeseries check to output-data/timeseries-checks/US4613088016
- manifest.json
- overview.md
- inputs/fund-static.json
- inputs/parameters.json
- inputs/fund-timeseries.csv
- outputs/source-comparison.csv
- outputs/source-comparison-summary.json
- outputs/consistency-report.txt
- outputs/regression.json
- outputs/regression-coefficients.csv
- outputs/regression-fit.csv
- outputs/regression-sample.csv
- outputs/lipper-regression-comparison.csv
- outputs/worst-mismatch.csv
- outputs/worst-mismatch-summary.json
- outputs/charts.html
- outputs/llm-evaluation-openai.json


PosixPath('/Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/output-data/timeseries-checks/US4613088016')